In [1]:
import os
import re
import sys
import time
import torch
import json
import psutil
import numpy as np
from pathlib import Path

# Ensure the project root is on sys.path so lib_v5 imports work
PROJECT_ROOT = os.path.dirname(os.path.abspath("__file__"))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from lib_v5.vr_network.model_param_init import ModelParameters

print("Imports OK")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Imports OK
PyTorch version: 2.4.1
CUDA available: False


In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
PTL_MODELS_DIR = os.path.join(PROJECT_ROOT, "ptl_models")
MODELPARAMS_DIR = os.path.join(PROJECT_ROOT, "lib_v5", "vr_network", "modelparams")

print(f"PTL models directory: {PTL_MODELS_DIR}")
print(f"Model params directory: {MODELPARAMS_DIR}")

# List all PTL model files
ptl_files = [f for f in os.listdir(PTL_MODELS_DIR) if f.endswith('.ptl')]
ptl_files.sort()

print(f"\nFound {len(ptl_files)} PTL model files:")
for f in ptl_files:
    size_mb = os.path.getsize(os.path.join(PTL_MODELS_DIR, f)) / (1024 * 1024)
    print(f"  {f} ({size_mb:.1f} MB)")

PTL models directory: /Users/nuhash/Desktop/other-codes/ultimatevocalremovergui/ptl_models
Model params directory: /Users/nuhash/Desktop/other-codes/ultimatevocalremovergui/lib_v5/vr_network/modelparams

Found 7 PTL model files:
  10_SP-UVR-2B-32000-1_vr.ptl (30.2 MB)
  11_SP-UVR-2B-32000-2_vr.ptl (30.2 MB)
  12_SP-UVR-3B-44100_vr.ptl (30.2 MB)
  13_SP-UVR-4B-44100-1_vr.ptl (30.2 MB)
  14_SP-UVR-4B-44100-2_vr.ptl (30.2 MB)
  15_SP-UVR-MID-44100-1_vr.ptl (30.2 MB)
  16_SP-UVR-MID-44100-2_vr.ptl (30.2 MB)


In [3]:
# ── Helper function to get model parameters ───────────────────────────────
def get_model_param_path(ptl_filename):
    """
    Infer the correct modelparams JSON for a .ptl file.
    
    Extracts srXXXX_hlYYYY from filename and matches to 1band_srXXXX_hlYYYY.json
    """
    # Extract sr and hl from filename like "MGM_HIGHEND_v4_sr44100_hl1024_nf2048_vr.ptl"
    match = re.search(r'sr(\d+)_hl(\d+)', ptl_filename)
    if match:
        sr = match.group(1)
        hl = match.group(2)
        param_file = f"1band_sr{sr}_hl{hl}.json"
        param_path = os.path.join(MODELPARAMS_DIR, param_file)
        if os.path.exists(param_path):
            return param_path
    
    # Fallback to common ones
    common_params = [
        "1band_sr44100_hl1024.json",
        "1band_sr44100_hl512.json", 
        "1band_sr32000_hl512.json",
        "1band_sr33075_hl384.json"
    ]
    
    for param_file in common_params:
        param_path = os.path.join(MODELPARAMS_DIR, param_file)
        if os.path.exists(param_path):
            return param_path
    
    raise FileNotFoundError(f"Could not find model parameters for {ptl_filename}")

# ── Benchmark function ──────────────────────────────────────────────────────
def benchmark_ptl_model(ptl_path, num_runs=5):
    """
    Benchmark a single PTL model.
    
    Returns dict with timing and memory stats.
    """
    model_name = os.path.basename(ptl_path)
    
    try:
        # Get model parameters
        param_path = get_model_param_path(model_name)
        mp = ModelParameters(param_path)
        bins = mp.param['bins']
        
        # Load model
        load_start = time.time()
        model = torch.jit.load(ptl_path, map_location="cpu")
        model.eval()
        load_time = time.time() - load_start
        
        # Create dummy input
        dummy_input = torch.rand(1, 2, bins, 128)
        
        # Warmup run
        with torch.no_grad():
            _ = model(dummy_input)
        
        # Benchmark runs
        times = []
        for _ in range(num_runs):
            start_time = time.time()
            with torch.no_grad():
                output = model(dummy_input)
            end_time = time.time()
            times.append(end_time - start_time)
        
        avg_time = np.mean(times)
        std_time = np.std(times)
        min_time = np.min(times)
        max_time = np.max(times)
        
        # Memory usage (approximate)
        model_size = os.path.getsize(ptl_path) / (1024 * 1024)  # MB
        
        return {
            'model_name': model_name,
            'load_time': load_time,
            'avg_inference_time': avg_time,
            'std_inference_time': std_time,
            'min_inference_time': min_time,
            'max_inference_time': max_time,
            'model_size_mb': model_size,
            'input_shape': tuple(dummy_input.shape),
            'output_shape': tuple(output.shape),
            'bins': bins,
            'success': True
        }
        
    except Exception as e:
        return {
            'model_name': model_name,
            'error': str(e),
            'success': False
        }

In [4]:
# ── Run benchmarks ─────────────────────────────────────────────────────────
print("Running benchmarks for PTL models...\n")

results = []
for ptl_file in ptl_files:
    ptl_path = os.path.join(PTL_MODELS_DIR, ptl_file)
    print(f"Benchmarking {ptl_file}...")
    
    result = benchmark_ptl_model(ptl_path, num_runs=5)
    results.append(result)
    
    if result['success']:
        print(f"  ✓ Success: {result['avg_inference_time']:.4f}s avg inference time")
    else:
        print(f"  ✗ Failed: {result['error']}")
    print()

print("Benchmark complete!")

Running benchmarks for PTL models...

Benchmarking 10_SP-UVR-2B-32000-1_vr.ptl...
  ✓ Success: 0.2003s avg inference time

Benchmarking 11_SP-UVR-2B-32000-2_vr.ptl...
  ✓ Success: 0.2200s avg inference time

Benchmarking 12_SP-UVR-3B-44100_vr.ptl...
  ✓ Success: 0.2011s avg inference time

Benchmarking 13_SP-UVR-4B-44100-1_vr.ptl...
  ✓ Success: 0.2008s avg inference time

Benchmarking 14_SP-UVR-4B-44100-2_vr.ptl...
  ✓ Success: 0.1960s avg inference time

Benchmarking 15_SP-UVR-MID-44100-1_vr.ptl...
  ✓ Success: 0.1831s avg inference time

Benchmarking 16_SP-UVR-MID-44100-2_vr.ptl...
  ✓ Success: 0.1967s avg inference time

Benchmark complete!


In [5]:
# show results in a table
import pandas as pd
df = pd.DataFrame(results)
df


,model_name,load_time,avg_inference_time,std_inference_time,min_inference_time,max_inference_time,model_size_mb,input_shape,output_shape,bins,success
0,10_SP-UVR-2B-32000-1_vr.ptl,0.046811,0.200325,0.018420,0.183170,0.232772,30.211884,"(1, 2, 1024, 128)","(1, 2, 673, 128)",1024,True
1,11_SP-UVR-2B-32000-2_vr.ptl,0.019254,0.220000,0.018552,0.201350,0.245574,30.211884,"(1, 2, 1024, 128)","(1, 2, 673, 128)",1024,True
2,12_SP-UVR-3B-44100_vr.ptl,0.023753,0.201069,0.011926,0.188910,0.223630,30.211637,"(1, 2, 1024, 128)","(1, 2, 673, 128)",1024,True
3,13_SP-UVR-4B-44100-1_vr.ptl,0.020532,0.200757,0.020309,0.182397,0.240080,30.211884,"(1, 2, 1024, 128)","(1, 2, 673, 128)",1024,True
4,14_SP-UVR-4B-44100-2_vr.ptl,0.019227,0.195958,0.009626,0.182034,0.208080,30.211884,"(1, 2, 1024, 128)","(1, 2, 673, 128)",1024,True
5,15_SP-UVR-MID-44100-1_vr.ptl,0.020956,0.183071,0.004271,0.178194,0.189397,30.211978,"(1, 2, 1024, 128)","(1, 2, 673, 128)",1024,True
6,16_SP-UVR-MID-44100-2_vr.ptl,0.023631,0.196717,0.003054,0.194222,0.202326,30.211978,"(1, 2, 1024, 128)","(1, 2, 673, 128)",1024,True


In [6]:
# Summary statistics
avg_times = [r['avg_inference_time'] for r in results]
avg_sizes = [r['model_size_mb'] for r in results]

print("\nSummary Statistics:")
print("=" * 50)
print(f"Total models tested: {len(results)}")
print(f"Successful: {len(results)}")
print(f"Failed: {len(results) - len(results)}")
print(f"Average inference time: {np.mean(avg_times):.4f}s")
print(f"Average model size: {np.mean(avg_sizes):.1f} MB")
print(f"Fastest model: {min(results, key=lambda x: x['avg_inference_time'])['model_name']} ({min(avg_times):.4f}s)")
print(f"Slowest model: {max(results, key=lambda x: x['avg_inference_time'])['model_name']} ({max(avg_times):.4f}s)")


Summary Statistics:
Total models tested: 7
Successful: 7
Failed: 0
Average inference time: 0.1997s
Average model size: 30.2 MB
Fastest model: 15_SP-UVR-MID-44100-1_vr.ptl (0.1831s)
Slowest model: 11_SP-UVR-2B-32000-2_vr.ptl (0.2200s)
